# Run FEWL on Google Colab

This notebook runs FEWL from `run.py` on one HaluEval split inside Colab, so your laptop can be off.
The code, data, API cache and logs all live in Google Drive, so they are kept after the Colab session ends:

```
MyDrive/HalluTrace/
├── HaluEvalData/<task>/<split>.jsonl                  the questions
├── APICache/                                          every OpenAI reply, so a restart does not pay twice
└── runs/fewl/<date>/<date>_<time>_<task>_<split>.log  one log per attempt
```

**Before the first run**, open the 🔑 Secrets panel in Colab's left sidebar, add these two secrets and switch on "Notebook access" for both:
- `OPENAI_API_KEY`: your OpenAI key (from the US data-residency project).
- `GITHUB_TOKEN`: a GitHub fine-grained token with read-only "Contents" access to `HalluTrace`, needed because the repo is private.

**To run:** choose the split in the settings cell, then use **Runtime → Run all**.
Keep the browser tab open. Free Colab stops after about 90 minutes without activity and after 12 hours at most.

**How long it takes:** about 7 seconds per question with `QUESTIONS_AT_ONCE = 5` in `run.py` (measured on the pilot).
- `sample_1000.jsonl`: about 2 hours.
- `test.jsonl` (9990 questions): about 20 hours at most, and less in practice, because answers to neighbour questions are reused from `APICache/`. Use Run all again in each new session to carry on; requests already answered come back for free.

The notebook writes your OpenAI key into `MyDrive/HalluTrace/.env` so `llm.py` can read it. That file is in your private Drive, and `.gitignore` keeps it out of git.

In [ ]:
from pathlib import Path

# Here we keep every setting in one place, so it is easy to see and change.
TASK_NAME = "qa"
SPLIT_FILE_NAME = "sample_1000.jsonl"
REPO_URL = "https://github.com/shoimya/HalluTrace.git"
DRIVE_FOLDER = Path("/content/drive")
REPO_FOLDER = DRIVE_FOLDER / "MyDrive" / "HalluTrace"
OPENAI_KEY_SECRET_NAME = "OPENAI_API_KEY"
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"
MAX_ATTEMPTS = 20
WAIT_SECONDS_BETWEEN_ATTEMPTS = 60

## Setup helpers
Small functions that connect Drive, fetch the code, write the key file and build the data.

In [ ]:
import subprocess
import sys

from google.colab import drive, userdata


def run_command(command):
    """Run a shell command, print what it says, and stop the notebook if it fails."""
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout + result.stderr)

    # Here we stop on a failure, so later cells do not run without code or data.
    result.check_returncode()


def read_secret(secret_name):
    """Read a value saved in Colab's Secrets panel."""
    return userdata.get(secret_name)


def build_clone_url(repo_url, github_token):
    """Put the GitHub token into the repo address, so the private repo can be downloaded."""
    return repo_url.replace("https://", f"https://{github_token}@")


def get_latest_code(repo_url, repo_folder, github_token):
    """Clone the repo into Drive the first time; in later sessions, pull the newest code instead."""
    clone_url = build_clone_url(repo_url, github_token)

    if (repo_folder / ".git").exists():
        print(f"Pulling the newest code into {repo_folder}")
        # Here we pass the token for this pull only, so it is never saved in the repo's settings.
        run_command(["git", "-C", str(repo_folder), "pull", "--ff-only", clone_url])
    else:
        print(f"Cloning the repo into {repo_folder}")
        run_command(["git", "clone", clone_url, str(repo_folder)])

        # Here we swap the saved address for the one without the token, so the token is not stored in Drive.
        run_command(["git", "-C", str(repo_folder), "remote", "set-url", "origin", repo_url])


def write_env_file(repo_folder, api_key):
    """Write the .env file that llm.py reads the OpenAI key from."""
    env_file = repo_folder / ".env"
    with open(env_file, "w", encoding="utf-8") as file:
        file.write(f"OPENAI_API_KEY={api_key}\n")
    print(f"Wrote {env_file}")


def prepare_dataset(repo_folder):
    """Download HaluEval and build the splits; the fixed seed gives the same splits as on the Mac."""
    dataset_script = repo_folder / "DatasetRetrievers" / "HaluEvaldataset.py"
    run_command([sys.executable, str(dataset_script)])

## Run helpers
`run_fewl_until_finished` starts the run again after any error, such as a dropped connection.
Each new attempt replays the finished questions from `APICache/` in a few minutes, then carries on.

In [ ]:
import contextlib
import time


class HideDetailLines:
    """Pass printed text on to the notebook, except the long "details:" lines, which the log file keeps anyway."""

    def __init__(self, notebook_output):
        """Remember where the notebook's printed text normally goes."""
        self.notebook_output = notebook_output
        self.hiding_a_line = False

    def write(self, text):
        """Show the text unless it belongs to a "details:" line."""
        # Here we drop the details text; print() sends the line ending separately, so we drop that next.
        if text.startswith("details:"):
            self.hiding_a_line = True
            return
        if self.hiding_a_line and text == "\n":
            self.hiding_a_line = False
            return
        self.notebook_output.write(text)

    def flush(self):
        """Show any waiting text straight away."""
        self.notebook_output.flush()


def run_fewl_until_finished(input_file, max_attempts, wait_seconds):
    """Run FEWL on the input file, starting again after any error, until it finishes."""
    import run

    for attempt_number in range(1, max_attempts + 1):
        log_path = run.build_log_path(run.METHOD_NAME, input_file)
        print(f"Attempt {attempt_number}/{max_attempts}, log: {log_path}")

        try:
            # Here we keep the notebook output short; every line still goes to the log file.
            with contextlib.redirect_stdout(HideDetailLines(sys.stdout)):
                run.run_fewl(input_file, log_path)
            print(f"Finished. Log saved to {log_path}")
            return log_path
        except Exception as error:
            # Here we wait and try again; the requests already answered come back from APICache/ for free.
            print(f"Attempt {attempt_number} stopped with: {error}")
            time.sleep(wait_seconds)

    raise RuntimeError(f"FEWL did not finish after {max_attempts} attempts; see the logs above.")

## Get ready
Connect Drive, fetch the newest code, write the key file and build the data.

In [ ]:
# Here we connect Drive and fetch the newest code from GitHub.
drive.mount(str(DRIVE_FOLDER))
github_token = read_secret(GITHUB_TOKEN_SECRET_NAME)
get_latest_code(REPO_URL, REPO_FOLDER, github_token)

# Here we give llm.py the OpenAI key, then download the data (skipped if it is already in Drive).
write_env_file(REPO_FOLDER, read_secret(OPENAI_KEY_SECRET_NAME))
prepare_dataset(REPO_FOLDER)

# Here we let Python find run.py and the files it imports.
sys.path.insert(0, str(REPO_FOLDER))

## Run FEWL
This cell runs for hours. It prints one line per question, and the accuracy table at the end.

In [ ]:
# Here we run FEWL on the chosen split until it finishes.
input_file = REPO_FOLDER / "HaluEvalData" / TASK_NAME / SPLIT_FILE_NAME
log_path = run_fewl_until_finished(input_file, MAX_ATTEMPTS, WAIT_SECONDS_BETWEEN_ATTEMPTS)

## Save to Drive
Drive saves files in the background. This makes sure the last logs and cache files are written before the session ends.

In [ ]:
# Here we wait for Drive to finish saving, then disconnect it.
drive.flush_and_unmount()
print(f"Saved. Your log is {log_path}")